Paso 1 — Extraer y normalizar datasets de turismo de datos.gov.co (Socrata/SODA).

Uso:
    pip install requests --break-system-packages
    python3 paso1_datos_gov.py

Salida:
    data/01_atractivos.json
    Lista de objetos con 5 llaves normalizadas:
    departamento, municipio, nombre, categoria, ubicacion



In [2]:
"""
Explorar datasets de datos.gov.co antes de meterlos a paso1_datos_gov.py

Usa la Socrata Discovery API para buscar datasets reales (evita el error
403 que da cuando se copia un ID de una URL tipo /w/xxxx-yyyy/ que es una
"vista/story" y no un dataset consultable).

Uso:
    python3 explorar_datasets.py
"""

import requests

DISCOVERY_URL = "https://api.us.socrata.com/api/catalog/v1"


def buscar_datasets(query: str, limit: int = 20) -> list:
    params = {
        "domains": "www.datos.gov.co",
        "q": query,
        "only": "datasets",
        "limit": limit,
    }
    resp = requests.get(DISCOVERY_URL, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json().get("results", [])


def inspeccionar_columnas(dataset_id: str):
    """Trae 1 fila real del dataset para ver los nombres de columna exactos."""
    url = f"https://www.datos.gov.co/resource/{dataset_id}.json"
    resp = requests.get(url, params={"$limit": 1}, timeout=30)
    resp.raise_for_status()
    rows = resp.json()
    if not rows:
        return []
    return list(rows[0].keys())


def main():
    resultados = buscar_datasets("divipola departamentos municipios", limit=20)

    print(f"Encontrados {len(resultados)} datasets para 'turismo':\n")
    for r in resultados:
        resource = r.get("resource", {})
        dataset_id = resource.get("id")
        nombre = resource.get("name")
        print(f"- {dataset_id}  |  {nombre}")

        try:
            columnas = inspeccionar_columnas(dataset_id)
            print(f"    columnas: {columnas}\n")
        except requests.RequestException as e:
            print(f"    !! no se pudo leer: {e}\n")


if __name__ == "__main__":
    main()

Encontrados 20 datasets para 'turismo':

- h2yr-zfb2  |  Subsidios De Vivienda Asignados
    columnas: ['departamento', 'c_digo_divipola_departamento', 'municipio', 'c_digo_divipola_municipio', 'programa', 'a_o_de_asignaci_n', 'estado_de_postulaci_n', 'hogares', 'valor_asignado']

- gdxc-w37w  |  DIVIPOLA- Códigos municipios
    !! no se pudo leer: 500 Server Error: Server Error for url: https://www.datos.gov.co/resource/gdxc-w37w.json?%24limit=1

- tcwu-r53g  |  Personas con acceso a agua potable y saneamiento básico por primera vez
    columnas: ['fecha_terminacion_proyecto', 'fecha_de_corte', 'c_digo_divipola_departamento', 'departamento', 'c_digo_divipola_municipio', 'municipio', 'indicador', 'valor', 'nombre_proyecto', 'origen', 'aporte_nacion', 'contrapartida', 'estado_seguimiento']

- vcjz-niiq  |  DIVIPOLA- Códigos departamentos
    columnas: ['codigo_departamento', 'nombre_departamento', 'longitud', 'latitud']

- xaxy-8nri  |  DIVIPOLA - Códigos cabeceras - Centros poblados
  

In [3]:
"""
Paso 1 (versión simple) — Listado oficial de departamentos y municipios de
Colombia (DIVIPOLA), dataset_id "xdk5-pm3f" en datos.gov.co.

No trae hoteles ni establecimientos — es solo la división político-
administrativa: cada departamento con sus municipios. Úsenlo como el
"esqueleto" sobre el cual luego cruzan clima, turismo, vuelos, etc.

Uso:
    pip install requests --break-system-packages
    python3 paso1_departamentos_municipios.py
"""

import json
import os
import requests

DATASET_ID = "gdxc-w37w"  # DIVIPOLA - Códigos municipios (confirmado activo)
SODA_URL = f"https://www.datos.gov.co/resource/{DATASET_ID}.json"
LIMIT = 5000


def fetch_all() -> list:
    all_rows = []
    offset = 0
    while True:
        params = {"$limit": LIMIT, "$offset": offset}
        resp = requests.get(SODA_URL, params=params, timeout=30)
        if resp.status_code == 404:
            raise RuntimeError(
                f"El dataset '{DATASET_ID}' ya no existe o cambió de ID "
                f"(404). Corre explorar_datasets.py con el query "
                f"'divipola departamentos municipios' para encontrar el "
                f"ID vigente y actualiza DATASET_ID en este script."
            )
        resp.raise_for_status()
        page = resp.json()
        if not page:
            break
        if offset == 0:
            print(f"Columnas reales del dataset: {list(page[0].keys())}")
            print("(si 'dpto'/'nom_mpio' no aparecen tal cual, ajusta las")
            print(" llaves usadas más abajo en normalize_row)\n")
        all_rows.extend(page)
        if len(page) < LIMIT:
            break
        offset += LIMIT
    return all_rows


def normalize_row(row: dict) -> dict:
    """Columnas confirmadas de gdxc-w37w:
    cod_dpto, dpto, cod_mpio, nom_mpio, tipo_municipio, longitud, latitud"""
    lower_map = {k.lower(): k for k in row.keys()}

    def get(*candidates):
        for c in candidates:
            if c in lower_map:
                return row[lower_map[c]]
        return None

    lat = get("latitud")
    lon = get("longitud")

    def to_float(value):
        if value in (None, ""):
            return None
        return float(str(value).replace(",", "."))

    return {
        "departamento": get("dpto"),
        "cod_departamento": get("cod_dpto"),
        "municipio": get("nom_mpio"),
        "cod_municipio": get("cod_mpio"),
        "tipo_municipio": get("tipo_municipio"),  # ej. "Municipio" / "Capital"
        "latitud": to_float(lat),
        "longitud": to_float(lon),
    }


def main():
    print(f"Descargando {DATASET_ID} (DIVIPOLA)...")
    try:
        rows = fetch_all()
    except RuntimeError as e:
        print(f"\n!! {e}")
        return
    print(f"{len(rows)} filas recibidas\n")

    normalizados = [normalize_row(r) for r in rows]

    # agrupa por departamento para que quede fácil de recorrer/validar
    por_departamento = {}
    for r in normalizados:
        depto = r["departamento"] or "SIN_DEPARTAMENTO"
        por_departamento.setdefault(depto, []).append({
            "municipio": r["municipio"],
            "latitud": r["latitud"],
            "longitud": r["longitud"],
        })

    os.makedirs("data", exist_ok=True)

    with open("data/00_municipios_por_departamento.json", "w", encoding="utf-8") as f:
        json.dump(por_departamento, f, ensure_ascii=False, indent=2)

    with open("data/00_divipola_plano.json", "w", encoding="utf-8") as f:
        json.dump(normalizados, f, ensure_ascii=False, indent=2)

    print(f"Listo: {len(por_departamento)} departamentos, {len(normalizados)} municipios")
    print("-> data/00_municipios_por_departamento.json (agrupado)")
    print("-> data/00_divipola_plano.json (lista plana)")
    if normalizados:
        print("\nEjemplo:")
        print(json.dumps(normalizados[0], ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()

Descargando gdxc-w37w (DIVIPOLA)...
Columnas reales del dataset: ['cod_dpto', 'dpto', 'cod_mpio', 'nom_mpio', 'tipo_municipio', 'longitud', 'latitud']
(si 'dpto'/'nom_mpio' no aparecen tal cual, ajusta las
 llaves usadas más abajo en normalize_row)

1122 filas recibidas

Listo: 33 departamentos, 1122 municipios
-> data/00_municipios_por_departamento.json (agrupado)
-> data/00_divipola_plano.json (lista plana)

Ejemplo:
{
  "departamento": "ANTIOQUIA",
  "cod_departamento": "05",
  "municipio": "MEDELLÍN",
  "cod_municipio": "05001",
  "tipo_municipio": "Municipio",
  "latitud": 6.246631,
  "longitud": -75.581775
}


In [4]:
"""
Paso 2 — Clima histórico mensual por departamento de Colombia.

Toma las capitales de departamento automáticamente desde el archivo que
generó el Paso 1 (data/00_divipola_plano.json), así que cubre los 32
departamentos sin necesidad de escribirlos a mano.

Usa Open-Meteo Archive API (gratis, sin API key) para promedios mensuales
de temperatura y precipitación de los últimos años, que es lo que
necesitas para decidir "la mejor época para viajar" (no un solo dato
puntual de clima actual).

Uso:
    pip install requests --break-system-packages
    python3 paso2_clima.py
"""

import json
import os
import statistics
from datetime import date
import requests

ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
YEARS_BACK = 5  # años de histórico a promediar


def cargar_capitales() -> dict:
    """Lee data/00_divipola_plano.json y elige, por departamento, el
    municipio marcado como capital (tipo_municipio contiene 'Capital').
    Si un departamento no tiene ninguno marcado así, usa el primer
    municipio de la lista como respaldo."""
    with open("data/00_divipola_plano.json", encoding="utf-8") as f:
        municipios = json.load(f)

    capitales = {}
    for m in municipios:
        depto = m["departamento"]
        es_capital = (m.get("tipo_municipio") or "").lower().find("capital") != -1
        if depto not in capitales or es_capital:
            capitales[depto] = m
            if es_capital:
                capitales[depto]["_confirmada"] = True

    faltan_confirmar = [d for d, m in capitales.items() if not m.get("_confirmada")]
    if faltan_confirmar:
        print("Aviso: estos departamentos no tenían 'Capital' explícita en "
              "tipo_municipio, se usó el primer municipio encontrado como "
              "aproximación (revisa si aplica):")
        for d in faltan_confirmar:
            print(f"  - {d} -> {capitales[d]['municipio']}")
        print()

    return capitales


def fetch_monthly_climate(lat: float, lon: float) -> dict:
    """Trae temperatura y precipitación diaria de los últimos N años y las
    agrupa por mes (1-12) para sacar promedios."""
    end = date.today()
    start = date(end.year - YEARS_BACK, 1, 1)

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start.isoformat(),
        "end_date": end.isoformat(),
        "daily": "temperature_2m_mean,precipitation_sum",
        "timezone": "America/Bogota",
    }
    resp = requests.get(ARCHIVE_URL, params=params, timeout=60)
    resp.raise_for_status()
    data = resp.json()

    by_month_temp = {m: [] for m in range(1, 13)}
    by_month_precip = {m: [] for m in range(1, 13)}

    dates = data["daily"]["time"]
    temps = data["daily"]["temperature_2m_mean"]
    precs = data["daily"]["precipitation_sum"]

    for d, t, p in zip(dates, temps, precs):
        month = int(d.split("-")[1])
        if t is not None:
            by_month_temp[month].append(t)
        if p is not None:
            by_month_precip[month].append(p)

    monthly = {}
    for m in range(1, 13):
        monthly[m] = {
            "temp_prom_c": round(statistics.mean(by_month_temp[m]), 1) if by_month_temp[m] else None,
            "precipitacion_prom_mm": round(statistics.mean(by_month_precip[m]), 1) if by_month_precip[m] else None,
        }
    return monthly


def main():
    capitales = cargar_capitales()
    print(f"{len(capitales)} departamentos detectados\n")

    result = {}
    for depto, info in capitales.items():
        municipio = info["municipio"]
        lat, lon = info["latitud"], info["longitud"]
        if lat is None or lon is None:
            print(f"-> {depto} ({municipio}): sin coordenadas, se omite")
            continue

        print(f"-> Clima histórico para {depto} ({municipio})...")
        try:
            monthly = fetch_monthly_climate(lat, lon)
        except requests.RequestException as e:
            print(f"   !! Error: {e}")
            continue

        result[depto] = {
            "municipio_referencia": municipio,
            "lat": lat,
            "lon": lon,
            "clima_mensual": monthly,
        }

    os.makedirs("data", exist_ok=True)
    out_path = "data/02_clima.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    print(f"\nListo -> {out_path}")


# -----------------------------------------------------------------------
# BLOQUE OPCIONAL — OpenWeather (clima actual / forecast 5 días)
# Descomenta y usa si tu materia pide específicamente esta API.
# Regístrate gratis en https://openweathermap.org/api
# -----------------------------------------------------------------------
# OPENWEATHER_API_KEY = "TU_API_KEY_AQUI"
#
# def fetch_current_weather(lat, lon):
#     url = "https://api.openweathermap.org/data/2.5/weather"
#     params = {"lat": lat, "lon": lon, "appid": OPENWEATHER_API_KEY, "units": "metric", "lang": "es"}
#     resp = requests.get(url, params=params, timeout=15)
#     resp.raise_for_status()
#     data = resp.json()
#     return {
#         "temp_actual_c": data["main"]["temp"],
#         "descripcion": data["weather"][0]["description"],
#         "timestamp": data["dt"],  # unix time
#     }


if __name__ == "__main__":
    main()

Aviso: estos departamentos no tenían 'Capital' explícita en tipo_municipio, se usó el primer municipio encontrado como aproximación (revisa si aplica):
  - ANTIOQUIA -> MEDELLÍN
  - ATLÁNTICO -> BARRANQUILLA
  - BOGOTÁ, D.C. -> BOGOTÁ, D.C.
  - BOLÍVAR -> CARTAGENA DE INDIAS
  - BOYACÁ -> TUNJA
  - CALDAS -> MANIZALES
  - CAQUETÁ -> FLORENCIA
  - CAUCA -> POPAYÁN
  - CESAR -> VALLEDUPAR
  - CÓRDOBA -> MONTERÍA
  - CUNDINAMARCA -> AGUA DE DIOS
  - CHOCÓ -> QUIBDÓ
  - HUILA -> NEIVA
  - LA GUAJIRA -> RIOHACHA
  - MAGDALENA -> SANTA MARTA
  - META -> VILLAVICENCIO
  - NARIÑO -> PASTO
  - NORTE DE SANTANDER -> SAN JOSÉ DE CÚCUTA
  - QUINDÍO -> ARMENIA
  - RISARALDA -> PEREIRA
  - SANTANDER -> BUCARAMANGA
  - SUCRE -> SINCELEJO
  - TOLIMA -> IBAGUÉ
  - VALLE DEL CAUCA -> SANTIAGO DE CALI
  - ARAUCA -> ARAUCA
  - CASANARE -> YOPAL
  - PUTUMAYO -> MOCOA
  - ARCHIPIÉLAGO DE SAN ANDRÉS, PROVIDENCIA Y SANTA CATALINA -> SAN ANDRÉS
  - AMAZONAS -> LETICIA
  - GUAINÍA -> INÍRIDA
  - GUAVIARE -> SAN

In [5]:
"""
Paso 3 — Unir municipios (Paso 1, DIVIPOLA) + clima mensual por
departamento (Paso 2) en un dataset estructurado, y calcular una primera
predicción heurística de "meses recomendados para viajar" (el baseline
sin ML que pide la sección 5 de la plantilla del proyecto).
"""

import json
import os

MESES_NOMBRE = {
    1: "enero", 2: "febrero", 3: "marzo", 4: "abril", 5: "mayo", 6: "junio",
    7: "julio", 8: "agosto", 9: "septiembre", 10: "octubre", 11: "noviembre", 12: "diciembre",
}

# Rango de temperatura considerado "agradable" para turismo — ajusta a tu criterio
TEMP_MIN_AGRADABLE = 18
TEMP_MAX_AGRADABLE = 28


def cargar_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def calcular_meses_recomendados(clima_mensual: dict, top_n: int = 3) -> list:
    """Heurística baseline: puntúa cada mes por temperatura dentro de rango
    agradable y baja precipitación, y devuelve los top_n meses."""
    scored = []
    for mes_str, valores in clima_mensual.items():
        mes = int(mes_str)
        temp = valores.get("temp_prom_c")
        precip = valores.get("precipitacion_prom_mm")
        if temp is None or precip is None:
            continue

        if TEMP_MIN_AGRADABLE <= temp <= TEMP_MAX_AGRADABLE:
            score_temp = 1.0
        else:
            dist = min(abs(temp - TEMP_MIN_AGRADABLE), abs(temp - TEMP_MAX_AGRADABLE))
            score_temp = max(0.0, 1.0 - dist / 10)

        score_precip = max(0.0, 1.0 - precip / 300)

        score = 0.6 * score_temp + 0.4 * score_precip
        scored.append((mes, round(score, 3)))

    scored.sort(key=lambda x: x[1], reverse=True)
    return [{"mes": MESES_NOMBRE[m], "score": s} for m, s in scored[:top_n]]


def main():
    municipios = cargar_json("data/00_divipola_plano.json")
    clima = cargar_json("data/02_clima.json")

    dataset_unido = []
    deptos_sin_clima = set()

    for m in municipios:
        depto = (m.get("departamento") or "").strip()
        clima_depto = clima.get(depto)

        registro = dict(m)
        if clima_depto:
            registro["clima_mensual"] = clima_depto["clima_mensual"]
            registro["meses_recomendados"] = calcular_meses_recomendados(
                clima_depto["clima_mensual"]
            )
        else:
            deptos_sin_clima.add(depto)
            registro["clima_mensual"] = None
            registro["meses_recomendados"] = None

        dataset_unido.append(registro)

    os.makedirs("data", exist_ok=True)
    out_path = "data/03_dataset_unido.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(dataset_unido, f, ensure_ascii=False, indent=2)

    print(f"Listo: {len(dataset_unido)} municipios -> {out_path}")
    if deptos_sin_clima:
        print(f"\nDepartamentos sin clima cargado:")
        for d in sorted(deptos_sin_clima):
            print(f"  - {d}")

    if dataset_unido:
        print("\nEjemplo de registro:")
        print(json.dumps(dataset_unido[0], ensure_ascii=False, indent=2))


main()

Listo: 1122 municipios -> data/03_dataset_unido.json

Ejemplo de registro:
{
  "departamento": "ANTIOQUIA",
  "cod_departamento": "05",
  "municipio": "MEDELLÍN",
  "cod_municipio": "05001",
  "tipo_municipio": "Municipio",
  "latitud": 6.246631,
  "longitud": -75.581775,
  "clima_mensual": {
    "1": {
      "temp_prom_c": 20.0,
      "precipitacion_prom_mm": 3.8
    },
    "2": {
      "temp_prom_c": 20.1,
      "precipitacion_prom_mm": 5.3
    },
    "3": {
      "temp_prom_c": 20.2,
      "precipitacion_prom_mm": 6.7
    },
    "4": {
      "temp_prom_c": 20.4,
      "precipitacion_prom_mm": 8.2
    },
    "5": {
      "temp_prom_c": 20.8,
      "precipitacion_prom_mm": 7.5
    },
    "6": {
      "temp_prom_c": 20.4,
      "precipitacion_prom_mm": 7.1
    },
    "7": {
      "temp_prom_c": 21.0,
      "precipitacion_prom_mm": 3.5
    },
    "8": {
      "temp_prom_c": 20.5,
      "precipitacion_prom_mm": 5.6
    },
    "9": {
      "temp_prom_c": 20.6,
      "precipitacion_prom_mm

In [20]:
"""
Paso 4 — Mejor mes para viajar por ruta, vía Travelpayouts Data API.

Genera rutas automáticamente desde un hub (Bogotá) hacia un catálogo de
aeropuertos comerciales de Colombia, en vez de escribirlas a mano una por
una. Cada resultado incluye el municipio/departamento del destino para
poder unirlo con data/03_dataset_unido.json en el Paso 5.

Importante: sin el parámetro 'month', el endpoint solo devuelve precios
cacheados recientes (todos caen en el mismo mes cercano), no un
comparativo real del año. Por eso aquí se consulta mes por mes de forma
explícita para los próximos 12 meses.
"""

import json
import os
import time
from datetime import date
import requests

from google.colab import userdata

TRAVELPAYOUTS_TOKEN = userdata.get("TRAVELPAYOUTS_TOKEN")

BASE_URL = "https://api.travelpayouts.com/v2/prices/month-matrix"

# -----------------------------------------------------------------------
# Catálogo de aeropuertos comerciales de Colombia (IATA -> municipio/depto).
# municipio/departamento deben coincidir (sin importar mayúsculas/tildes)
# con los valores de data/00_divipola_plano.json para que el Paso 5 los
# pueda unir. Agrega/quita destinos según lo que tu proyecto recomiende.
# -----------------------------------------------------------------------
AEROPUERTOS = {
    "BOG": {"municipio": "BOGOTA D.C.", "departamento": "BOGOTA"},          # hub
    "CTG": {"municipio": "CARTAGENA DE INDIAS", "departamento": "BOLIVAR"},
    "SMR": {"municipio": "SANTA MARTA", "departamento": "MAGDALENA"},
    "ADZ": {"municipio": "SAN ANDRES",  "departamento": "SAN ANDRES, PROVIDENCIA Y SANTA CATALINA"},
    "MDE": {"municipio": "MEDELLIN",    "departamento": "ANTIOQUIA"},
    "CLO": {"municipio": "SANTIAGO DE CALI", "departamento": "VALLE DEL CAUCA"},
    "BAQ": {"municipio": "BARRANQUILLA","departamento": "ATLANTICO"},
    "PEI": {"municipio": "PEREIRA",     "departamento": "RISARALDA"},
    "ARM": {"municipio": "ARMENIA",     "departamento": "QUINDIO"},
    "MZL": {"municipio": "MANIZALES",   "departamento": "CALDAS"},
    "RCH": {"municipio": "RIOHACHA",    "departamento": "LA GUAJIRA"},
    "LET": {"municipio": "LETICIA",     "departamento": "AMAZONAS"},
    "CUC": {"municipio": "SAN JOSE DE CUCUTA", "departamento": "NORTE DE SANTANDER"},
    "IBE": {"municipio": "IBAGUE",      "departamento": "TOLIMA"},
    "PSO": {"municipio": "PASTO",       "departamento": "NARIÑO"},
    "MTR": {"municipio": "MONTERIA",    "departamento": "CORDOBA"},
    "NVA": {"municipio": "NEIVA",       "departamento": "HUILA"},
    "VVC": {"municipio": "VILLAVICENCIO","departamento": "META"},
    "PPN": {"municipio": "POPAYAN",     "departamento": "CAUCA"},
    "TCO": {"municipio": "TUNJA",       "departamento": "BOYACA"},  # aprox. (Tunja no tiene aeropuerto comercial grande)
}

HUBS = ["BOG", "MDE", "CLO", "BAQ"]  # ciudades de origen más comunes en Colombia
RUTAS = [
    {"origen": hub, "destino": iata}
    for hub in HUBS
    for iata in AEROPUERTOS
    if iata != hub
]

MESES_A_CONSULTAR = 12  # próximos 12 meses desde hoy

print(f"Se van a consultar {len(RUTAS)} rutas x {MESES_A_CONSULTAR} meses "
      f"= {len(RUTAS) * MESES_A_CONSULTAR} llamadas. Puede tardar bastante "
      f"(reduce HUBS o MESES_A_CONSULTAR si quieres una corrida más rápida).")


def proximos_meses(n: int) -> list:
    """YYYY-MM-01 para los próximos n meses, empezando por el mes actual."""
    hoy = date.today()
    meses = []
    y, m = hoy.year, hoy.month
    for _ in range(n):
        meses.append(f"{y:04d}-{m:02d}-01")
        m += 1
        if m > 12:
            m = 1
            y += 1
    return meses


def precio_para_mes(origen: str, destino: str, mes: str):
    headers = {"x-access-token": TRAVELPAYOUTS_TOKEN}
    params = {
        "currency": "cop",
        "origin": origen,
        "destination": destino,
        "month": mes,
        "show_to_affiliates": "true",
    }
    resp = requests.get(BASE_URL, headers=headers, params=params, timeout=30)
    resp.raise_for_status()
    body = resp.json()
    if not body.get("success"):
        raise RuntimeError(body.get("error", "error desconocido"))
    datos = body.get("data", [])
    if not datos:
        return None
    return min(d["value"] for d in datos if "value" in d)


def main():
    resultados = []

    for ruta in RUTAS:
        print(f"-> {ruta['origen']} -> {ruta['destino']}")
        precios_por_mes = []

        for mes in proximos_meses(MESES_A_CONSULTAR):
            try:
                precio = precio_para_mes(ruta["origen"], ruta["destino"], mes)
            except (requests.RequestException, RuntimeError) as e:
                print(f"   !! Error en {mes[:7]}: {e}")
                continue

            if precio is not None:
                print(f"   {mes[:7]}: {precio:,.0f} COP")
                precios_por_mes.append({"mes": mes[:7], "precio": precio})
            else:
                print(f"   {mes[:7]}: sin datos")

            time.sleep(0.3)  # evita saturar el rate limit

        if not precios_por_mes:
            print("   (sin datos para ningún mes en esta ruta)")
            continue

        mas_barato = min(precios_por_mes, key=lambda x: x["precio"])
        destino_info = AEROPUERTOS.get(ruta["destino"], {})
        resultados.append({
            "origen": ruta["origen"],
            "destino": ruta["destino"],
            "destino_municipio": destino_info.get("municipio"),
            "destino_departamento": destino_info.get("departamento"),
            "precios_por_mes": precios_por_mes,
            "mes_mas_barato": mas_barato["mes"],
            "precio_mas_barato": mas_barato["precio"],
        })

    os.makedirs("data", exist_ok=True)
    out_path = "data/04_vuelos.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(resultados, f, ensure_ascii=False, indent=2)

    print(f"\nListo -> {out_path}")
    if resultados:
        print("\nEjemplo de registro:")
        print(json.dumps(resultados[0], ensure_ascii=False, indent=2))


main()

Se van a consultar 76 rutas x 12 meses = 912 llamadas. Puede tardar bastante (reduce HUBS o MESES_A_CONSULTAR si quieres una corrida más rápida).
-> BOG -> CTG
   2026-08: 48,443 COP
   2026-09: 56,255 COP
   2026-10: sin datos
   2026-11: sin datos
   2026-12: sin datos
   2027-01: sin datos
   2027-02: 231,540 COP
   2027-03: sin datos
   2027-04: sin datos
   2027-05: sin datos
   2027-06: sin datos
   2027-07: sin datos
-> BOG -> SMR
   2026-08: 117,106 COP
   2026-09: 110,113 COP
   2026-10: 212,605 COP
   2026-11: sin datos
   2026-12: 495,764 COP
   2027-01: sin datos
   2027-02: sin datos
   2027-03: sin datos
   2027-04: sin datos
   2027-05: sin datos
   2027-06: sin datos
   2027-07: sin datos
-> BOG -> ADZ
   2026-08: 207,792 COP
   2026-09: sin datos
   2026-10: sin datos
   2026-11: 154,158 COP
   2026-12: sin datos
   2027-01: sin datos
   2027-02: sin datos
   2027-03: sin datos
   2027-04: 271,839 COP
   2027-05: sin datos
   2027-06: sin datos
   2027-07: sin datos
->

In [7]:
"""
Paso 5 — Unificar data/03_dataset_unido.json (municipios + clima) con
data/04_vuelos.json (precios por mes) en un solo dataset final por
destino, listo para alimentar el modelo del M1.

Solo los municipios que SÍ tienen vuelo consultado en el Paso 4 quedan
con info de precios; el resto queda solo con clima (útil para otras
partes del proyecto, pero sin señal de precio).
"""

import json
import os


def normalizar(texto):
    """Para comparar nombres sin pelearse con mayúsculas/tildes/puntuación."""
    if not texto:
        return ""
    reemplazos = str.maketrans("ÁÉÍÓÚÑ", "AEIOUN")
    limpio = texto.upper().translate(reemplazos)
    limpio = limpio.replace(",", "").replace(".", "")
    return " ".join(limpio.split())  # colapsa espacios múltiples


def calcular_mejor_epoca_combinada(meses_recomendados_clima, vuelos):
    """Cruza los mejores meses por clima con el mes más barato encontrado
    entre TODOS los orígenes disponibles para ese destino. Heurística
    simple: si el mes más barato también aparece en los meses
    recomendados por clima, es la mejor opción posible (barato + buen
    clima). Si no, se reportan ambos por separado para que el usuario
    decida el trade-off."""
    if not meses_recomendados_clima or not vuelos:
        return None

    meses_clima = {m["mes"] for m in meses_recomendados_clima}
    mas_barato_global = min(vuelos, key=lambda v: v["precio_mas_barato"])
    mes_barato = mas_barato_global["mes_mas_barato"]

    return {
        "coincide_barato_y_buen_clima": mes_barato in meses_clima,
        "mejor_origen": mas_barato_global["origen"],
        "mes_mas_barato": mes_barato,
        "precio_mas_barato": mas_barato_global["precio_mas_barato"],
        "mejores_meses_clima": sorted(meses_clima),
    }


def main():
    with open("data/03_dataset_unido.json", encoding="utf-8") as f:
        municipios = json.load(f)

    with open("data/04_vuelos.json", encoding="utf-8") as f:
        vuelos = json.load(f)

    # ahora puede haber varias rutas (distintos orígenes) hacia el mismo
    # destino, así que se agrupan en una lista en vez de sobreescribir.
    # La clave es (municipio, departamento) y NO solo municipio, porque
    # Colombia tiene municipios homónimos en departamentos distintos
    # (ej. "San Andrés" existe en el archipiélago Y en Santander).
    vuelos_por_municipio = {}
    for v in vuelos:
        key = (normalizar(v.get("destino_municipio")), normalizar(v.get("destino_departamento")))
        if key[0]:
            vuelos_por_municipio.setdefault(key, []).append(v)

    final = []
    con_vuelo = 0
    for m in municipios:
        key = (normalizar(m.get("municipio")), normalizar(m.get("departamento")))
        vuelos_del_destino = vuelos_por_municipio.get(key)

        registro = dict(m)
        registro["vuelos_por_origen"] = vuelos_del_destino  # None si no hay ruta consultada
        registro["mejor_epoca_combinada"] = calcular_mejor_epoca_combinada(
            m.get("meses_recomendados"), vuelos_del_destino
        )
        if vuelos_del_destino:
            con_vuelo += 1
        final.append(registro)

    os.makedirs("data", exist_ok=True)
    out_path = "data/05_dataset_final.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(final, f, ensure_ascii=False, indent=2)

    print(f"Listo: {len(final)} municipios totales, {con_vuelo} con datos "
          f"de vuelo -> {out_path}")

    # muestra un ejemplo de un municipio que SÍ tenga vuelo, para verificar
    ejemplo = next((r for r in final if r["vuelos_por_origen"]), None)
    if ejemplo:
        print("\nEjemplo de registro con vuelo:")
        print(json.dumps(ejemplo, ensure_ascii=False, indent=2))
    else:
        print("\nAviso: ningún municipio hizo match con los vuelos del "
              "Paso 4. Revisa que 'destino_municipio' en data/04_vuelos.json "
              "coincida con los nombres de municipio del Paso 1 (imprime "
              "ambos y compara si hace falta ajustar AEROPUERTOS).")


main()

Listo: 1122 municipios totales, 14 con datos de vuelo -> data/05_dataset_final.json

Ejemplo de registro con vuelo:
{
  "departamento": "ANTIOQUIA",
  "cod_departamento": "05",
  "municipio": "MEDELLÍN",
  "cod_municipio": "05001",
  "tipo_municipio": "Municipio",
  "latitud": 6.246631,
  "longitud": -75.581775,
  "clima_mensual": {
    "1": {
      "temp_prom_c": 20.0,
      "precipitacion_prom_mm": 3.8
    },
    "2": {
      "temp_prom_c": 20.1,
      "precipitacion_prom_mm": 5.3
    },
    "3": {
      "temp_prom_c": 20.2,
      "precipitacion_prom_mm": 6.7
    },
    "4": {
      "temp_prom_c": 20.4,
      "precipitacion_prom_mm": 8.2
    },
    "5": {
      "temp_prom_c": 20.8,
      "precipitacion_prom_mm": 7.5
    },
    "6": {
      "temp_prom_c": 20.4,
      "precipitacion_prom_mm": 7.1
    },
    "7": {
      "temp_prom_c": 21.0,
      "precipitacion_prom_mm": 3.5
    },
    "8": {
      "temp_prom_c": 20.5,
      "precipitacion_prom_mm": 5.6
    },
    "9": {
      "temp_pr

In [8]:
import json

AEROPUERTOS_CORREGIDO = {
    "CTG": {"municipio": "CARTAGENA DE INDIAS", "departamento": "BOLIVAR"},
    "CLO": {"municipio": "SANTIAGO DE CALI", "departamento": "VALLE DEL CAUCA"},
    "CUC": {"municipio": "SAN JOSE DE CUCUTA", "departamento": "NORTE DE SANTANDER"},
}

with open("data/04_vuelos.json", encoding="utf-8") as f:
    vuelos = json.load(f)

corregidos = 0
for v in vuelos:
    fix = AEROPUERTOS_CORREGIDO.get(v["destino"])
    if fix:
        v["destino_municipio"] = fix["municipio"]
        v["destino_departamento"] = fix["departamento"]
        corregidos += 1

with open("data/04_vuelos.json", "w", encoding="utf-8") as f:
    json.dump(vuelos, f, ensure_ascii=False, indent=2)

print(f"Corregidos {corregidos} registros")

Corregidos 8 registros


In [13]:
"""
Paso 6 — Hoteles, restaurantes y atractivos por destino, vía OpenStreetMap
Overpass API. Reintenta con espera en 429, cambia de espejo en otros
errores, y guarda progreso incremental (puedes parar y retomar).
"""

import json
import os
import time
import requests

OVERPASS_URLS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.openstreetmap.ru/cgi/interpreter",
]

HEADERS = {
    "User-Agent": "TravelGenie-EAFIT-ProyectoAcademico/1.0 (uso educativo)",
    "Accept": "application/json",
}

RADIO_METROS = 15000

CATEGORIAS = {
    "tourism": ["hotel", "guest_house", "attraction", "museum", "viewpoint"],
    "amenity": ["restaurant"],
}


def construir_query(lat, lon, radio=RADIO_METROS):
    partes = []
    for key, valores in CATEGORIAS.items():
        patron = "|".join(valores)
        partes.append(f'node["{key}"~"^({patron})$"](around:{radio},{lat},{lon});')
        partes.append(f'way["{key}"~"^({patron})$"](around:{radio},{lat},{lon});')
    cuerpo = "\n  ".join(partes)
    return f"""
[out:json][timeout:60];
(
  {cuerpo}
);
out center tags;
"""


def consultar_overpass(lat, lon):
    query = construir_query(lat, lon)
    ultimo_error = None
    for url in OVERPASS_URLS:
        for intento in range(2):
            try:
                resp = requests.post(url, data={"data": query}, headers=HEADERS, timeout=90)
                if resp.status_code == 429:
                    espera = 30
                    print(f"   (429 en {url}, esperando {espera}s antes de reintentar...)")
                    time.sleep(espera)
                    continue
                resp.raise_for_status()
                return resp.json().get("elements", [])
            except requests.RequestException as e:
                ultimo_error = e
                print(f"   (falló {url}: {e})")
                break
        continue
    raise ultimo_error if ultimo_error else RuntimeError("todos los servidores fallaron")


def normalizar_elemento(el):
    tags = el.get("tags", {})
    if el["type"] == "node":
        lat, lon = el.get("lat"), el.get("lon")
    else:
        centro = el.get("center", {})
        lat, lon = centro.get("lat"), centro.get("lon")

    return {
        "nombre": tags.get("name"),
        "categoria": tags.get("tourism") or tags.get("amenity"),
        "lat": lat,
        "lon": lon,
        "direccion": tags.get("addr:street"),
    }


def main():
    with open("data/05_dataset_final.json", encoding="utf-8") as f:
        municipios = json.load(f)

    destinos = [m for m in municipios if m.get("vuelos_por_origen")]
    print(f"Consultando lugares para {len(destinos)} destinos "
          f"(radio {RADIO_METROS/1000:.0f} km)...")

    os.makedirs("data", exist_ok=True)
    out_path = "data/06_lugares.json"

    try:
        with open(out_path, encoding="utf-8") as f:
            resultado = json.load(f)
        print(f"Retomando: {len(resultado)} destinos ya guardados de una corrida previa.")
    except FileNotFoundError:
        resultado = {}

    for m in destinos:
        nombre = m["municipio"]
        if nombre in resultado:
            print(f"-> {nombre} (ya estaba guardado, se omite)")
            continue

        print(f"-> {nombre}")
        try:
            elementos = consultar_overpass(m["latitud"], m["longitud"])
        except requests.RequestException as e:
            print(f"   !! Error final: {e}")
            continue

        lugares = [normalizar_elemento(el) for el in elementos]
        lugares = [l for l in lugares if l["nombre"]]

        hoteles = [l for l in lugares if l["categoria"] in ("hotel", "guest_house")]
        restaurantes = [l for l in lugares if l["categoria"] == "restaurant"]
        atractivos = [l for l in lugares if l["categoria"] in ("attraction", "museum", "viewpoint")]

        resultado[nombre] = {
            "hoteles": hoteles,
            "restaurantes": restaurantes,
            "atractivos": atractivos,
        }
        print(f"   {len(hoteles)} hoteles, {len(restaurantes)} restaurantes, "
              f"{len(atractivos)} atractivos")

        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(resultado, f, ensure_ascii=False, indent=2)

        time.sleep(5)

    print(f"\nListo: {len(resultado)}/{len(destinos)} destinos -> {out_path}")


main()

Consultando lugares para 14 destinos (radio 15 km)...
Retomando: 13 destinos ya guardados de una corrida previa.
-> MEDELLÍN (ya estaba guardado, se omite)
-> BARRANQUILLA (ya estaba guardado, se omite)
-> TUNJA (ya estaba guardado, se omite)
-> MANIZALES
   46 hoteles, 120 restaurantes, 15 atractivos
-> POPAYÁN (ya estaba guardado, se omite)
-> MONTERÍA (ya estaba guardado, se omite)
-> NEIVA (ya estaba guardado, se omite)
-> RIOHACHA (ya estaba guardado, se omite)
-> SANTA MARTA (ya estaba guardado, se omite)
-> VILLAVICENCIO (ya estaba guardado, se omite)
-> PASTO (ya estaba guardado, se omite)
-> PEREIRA (ya estaba guardado, se omite)
-> IBAGUÉ (ya estaba guardado, se omite)
-> LETICIA (ya estaba guardado, se omite)

Listo: 14/14 destinos -> data/06_lugares.json


In [14]:
"""
Paso 7 (v2) — Generar el dataset de fine-tuning con una reformulación
importante de la tarea:

ANTES: input = pregunta sobre un destino -> output = recomendación (el
modelo tenía que "recordar" qué hotel/restaurante corresponde a cada
ciudad solo por el nombre; con ~300 ejemplos y decenas de nombres únicos,
eso es un problema de memorización que no se resuelve con más épocas).

AHORA: input = pregunta del usuario + LOS HECHOS YA DADOS (clima, precio,
hoteles, restaurantes, atractivos) -> output = la misma recomendación en
lenguaje natural. La tarea pasa de "recordar hechos" a "redactar hechos
que ya te dieron" — mucho más aprendible con pocos ejemplos, y además es
como funcionaría el sistema real (primero se consultan los datos con
código determinístico, el modelo solo se usa para la redacción final).

Uso:
    python3 paso7_generar_pares_entrenamiento.py
"""

import json
import os
import random

random.seed(42)  # reproducibilidad (la rúbrica la pide explícitamente)

# -----------------------------------------------------------------------
# Plantillas de PREGUNTA (lo que el usuario escribe; se combina con los
# hechos para formar el input completo)
# -----------------------------------------------------------------------
PREGUNTAS_GENERICAS = [
    "Quiero viajar por Colombia con buen clima y vuelos económicos.",
    "Busco un destino turístico barato para visitar pronto.",
    "Recomiéndame un lugar en Colombia con buen clima y precio accesible.",
    "¿A dónde me recomiendas viajar si quiero ahorrar en el tiquete?",
    "Necesito un plan de viaje económico con buen clima.",
    "Dame una recomendación de destino turístico en Colombia.",
    "Estoy planeando un viaje, ¿qué destino en Colombia me recomiendas?",
    "Quiero unas vacaciones baratas en Colombia, ¿alguna sugerencia?",
]

PREGUNTAS_ESPECIFICAS = [
    "¿Cuál es el mejor mes para viajar a {municipio}?",
    "Dame una recomendación de viaje a {municipio}.",
    "Quiero ir a {municipio}, ¿cuándo me conviene viajar?",
    "Háblame de {municipio} como destino turístico.",
    "¿Vale la pena viajar a {municipio}? ¿En qué mes?",
    "¿Qué tan caro es volar a {municipio}?",
    "Cuéntame sobre viajar a {municipio}.",
    "¿Me recomiendas {municipio} para vacacionar?",
    "Estoy pensando en ir a {municipio}, ¿qué me dices?",
    "¿Cómo es viajar a {municipio}?",
]

PREGUNTAS_HOSPEDAJE = [
    "¿Dónde me puedo hospedar en {municipio}?",
    "Recomiéndame hoteles en {municipio}.",
    "¿Qué opciones de alojamiento hay en {municipio}?",
]

PREGUNTAS_COMIDA = [
    "¿Dónde puedo comer en {municipio}?",
    "Recomiéndame restaurantes en {municipio}.",
    "¿Qué comida probar si voy a {municipio}?",
]

PREGUNTAS_ATRACTIVOS = [
    "¿Qué puedo visitar en {municipio}?",
    "¿Cuáles son los atractivos turísticos de {municipio}?",
    "¿Qué no me puedo perder si voy a {municipio}?",
]

# -----------------------------------------------------------------------
# Fragmentos de SALIDA (frases con variación; se combinan con los MISMOS
# datos que se muestran en el input, así el modelo aprende a redactar,
# no a inventar)
# -----------------------------------------------------------------------
APERTURAS = [
    "Te recomiendo viajar a {municipio}, {departamento}.",
    "Una buena opción es {municipio}, en el departamento de {departamento}.",
    "{municipio} ({departamento}) es un destino que te puede interesar.",
]

CLIMA_FRASES = [
    "El mejor mes por clima es {mes_clima}, con temperaturas agradables y poca lluvia.",
    "Por clima, {mes_clima} es la mejor época para visitarlo.",
    "El clima es más favorable en {mes_clima}.",
]

VUELO_FRASES = [
    "El vuelo más económico encontrado fue en {mes_barato}, desde {origen}, por ${precio:,.0f} COP.",
    "Si buscas ahorrar, vuela en {mes_barato} desde {origen}: encontramos tiquetes desde ${precio:,.0f} COP.",
    "El precio más bajo detectado fue ${precio:,.0f} COP volando desde {origen} en {mes_barato}.",
]

COINCIDENCIA_FRASES = {
    True: "Además, ese mes económico coincide con uno de los mejores momentos climáticos del año — combinación ideal.",
    False: "Ten en cuenta que el mes más barato no coincide con el mejor clima, así que es un trade-off entre precio y clima.",
}

VERBO_LUGARES = {
    "hoteles": "Para hospedarte puedes considerar",
    "restaurantes": "Para comer, algunas opciones son",
    "atractivos": "No te pierdas",
}

MESES_NOMBRE = {
    "01": "enero", "02": "febrero", "03": "marzo", "04": "abril",
    "05": "mayo", "06": "junio", "07": "julio", "08": "agosto",
    "09": "septiembre", "10": "octubre", "11": "noviembre", "12": "diciembre",
}


def mes_legible(valor: str) -> str:
    if valor and "-" in valor and valor.split("-")[-1] in MESES_NOMBRE:
        return MESES_NOMBRE[valor.split("-")[-1]]
    return valor


def preparar_datos(destino: dict, lugares: dict) -> dict:
    """Muestrea UNA vez los datos de este destino (hoteles/restaurantes/
    atractivos incluidos). Este mismo diccionario se usa para construir
    el input (hechos dados) y el output (redacción), así quedan
    garantizadamente consistentes entre sí."""
    combinada = destino["mejor_epoca_combinada"]

    def muestra(categoria, k=3):
        items = lugares.get(categoria, []) if lugares else []
        elegidos = random.sample(items, k=min(k, len(items))) if items else []
        return [i["nombre"] for i in elegidos if i.get("nombre")]

    return {
        "municipio": destino["municipio"].title(),
        "departamento": destino["departamento"].title(),
        "mes_clima": combinada["mejores_meses_clima"][0],
        "mes_barato": mes_legible(combinada["mes_mas_barato"]),
        "origen": combinada["mejor_origen"],
        "precio": combinada["precio_mas_barato"],
        "coincide": combinada["coincide_barato_y_buen_clima"],
        "hoteles": muestra("hoteles"),
        "restaurantes": muestra("restaurantes"),
        "atractivos": muestra("atractivos"),
    }


def construir_bloque_hechos(datos: dict) -> str:
    """Serializa los datos como texto simple para meter en el INPUT."""
    partes = [
        f"Destino: {datos['municipio']}, {datos['departamento']}.",
        f"Mejor mes por clima: {datos['mes_clima']}.",
        f"Vuelo más económico: {datos['mes_barato']} desde {datos['origen']}, "
        f"${datos['precio']:,.0f} COP.",
    ]
    for categoria in ("hoteles", "restaurantes", "atractivos"):
        nombres = datos[categoria]
        if nombres:
            partes.append(f"{categoria.capitalize()}: {', '.join(nombres)}.")
    return " ".join(partes)


def construir_salida(datos: dict, categorias_a_mencionar: list = None) -> str:
    """Redacta la recomendación en lenguaje natural usando los MISMOS
    datos que ya se mostraron en el input. Si categorias_a_mencionar es
    None, menciona todas las categorías con datos disponibles."""
    partes = [
        random.choice(APERTURAS).format(municipio=datos["municipio"], departamento=datos["departamento"]),
        random.choice(CLIMA_FRASES).format(mes_clima=datos["mes_clima"]),
        random.choice(VUELO_FRASES).format(
            mes_barato=datos["mes_barato"], origen=datos["origen"], precio=datos["precio"],
        ),
        COINCIDENCIA_FRASES[datos["coincide"]],
    ]

    categorias = categorias_a_mencionar or ["hoteles", "restaurantes", "atractivos"]
    for categoria in categorias:
        nombres = datos[categoria]
        if nombres:
            partes.append(f"{VERBO_LUGARES[categoria]}: {', '.join(nombres)}.")

    return " ".join(partes)


def construir_par(pregunta: str, datos: dict, categorias_a_mencionar: list = None) -> dict:
    hechos = construir_bloque_hechos(datos)
    entrada = f"Datos: {hechos} Pregunta: {pregunta}"
    salida = construir_salida(datos, categorias_a_mencionar)
    return {"input": entrada, "output": salida}


def main():
    with open("data/05_dataset_final.json", encoding="utf-8") as f:
        municipios = json.load(f)

    try:
        with open("data/06_lugares.json", encoding="utf-8") as f:
            lugares_por_municipio = json.load(f)
    except FileNotFoundError:
        print("Aviso: no encontré data/06_lugares.json, sigo sin esa info.")
        lugares_por_municipio = {}

    destinos = [m for m in municipios if m.get("mejor_epoca_combinada")]
    print(f"Generando pares para {len(destinos)} destinos...")

    pares = []
    for destino in destinos:
        municipio_nombre = destino["municipio"]
        lugares = lugares_por_municipio.get(municipio_nombre, {})
        municipio_titulo = municipio_nombre.title()

        # preguntas generales (info completa)
        for plantilla in PREGUNTAS_ESPECIFICAS:
            datos = preparar_datos(destino, lugares)  # muestreo propio por ejemplo
            pregunta = plantilla.format(municipio=municipio_titulo)
            pares.append(construir_par(pregunta, datos))

        # preguntas enfocadas por categoría
        for plantillas, categoria in [
            (PREGUNTAS_HOSPEDAJE, "hoteles"),
            (PREGUNTAS_COMIDA, "restaurantes"),
            (PREGUNTAS_ATRACTIVOS, "atractivos"),
        ]:
            for plantilla in plantillas:
                datos = preparar_datos(destino, lugares)
                if not datos[categoria]:
                    continue  # este destino no tiene datos de esa categoría
                pregunta = plantilla.format(municipio=municipio_titulo)
                pares.append(construir_par(pregunta, datos, categorias_a_mencionar=[categoria]))

    # preguntas genéricas: se empareja con varios destinos al azar
    for pregunta in PREGUNTAS_GENERICAS:
        for _ in range(10):
            destino = random.choice(destinos)
            municipio_nombre = destino["municipio"]
            lugares = lugares_por_municipio.get(municipio_nombre, {})
            datos = preparar_datos(destino, lugares)
            pares.append(construir_par(pregunta, datos))

    random.shuffle(pares)

    corte = int(len(pares) * 0.8)
    train, val = pares[:corte], pares[corte:]

    os.makedirs("data", exist_ok=True)

    with open("data/07_train.jsonl", "w", encoding="utf-8") as f:
        for p in train:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")

    with open("data/07_val.jsonl", "w", encoding="utf-8") as f:
        for p in val:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")

    print(f"\nTotal pares: {len(pares)} (train: {len(train)}, val: {len(val)})")
    print("-> data/07_train.jsonl")
    print("-> data/07_val.jsonl")

    print("\nEjemplo:")
    print(json.dumps(pares[0], ensure_ascii=False, indent=2))


main()

Generando pares para 14 destinos...

Total pares: 346 (train: 276, val: 70)
-> data/07_train.jsonl
-> data/07_val.jsonl

Ejemplo:
{
  "input": "Datos: Destino: Montería, Córdoba. Mejor mes por clima: diciembre. Vuelo más económico: agosto desde BOG, $122,867 COP. Hoteles: Hotel La Cabaña, Hostal Meridiano, GHL Hotel Monteria. Restaurantes: Restaurante Comidas Rápidas el Lobo, Mi Casa, Asados y arepas R8. Atractivos: Piscinas Ecológicas Las Doncellas, Pasaje del Sol, Estadio de futbol jaraguay de monteria. Pregunta: Háblame de Montería como destino turístico.",
  "output": "Una buena opción es Montería, en el departamento de Córdoba. Por clima, diciembre es la mejor época para visitarlo. El precio más bajo detectado fue $122,867 COP volando desde BOG en agosto. Ten en cuenta que el mes más barato no coincide con el mejor clima, así que es un trade-off entre precio y clima. Para hospedarte puedes considerar: Hotel La Cabaña, Hostal Meridiano, GHL Hotel Monteria. Para comer, algunas opcio

# **Inicio fine-tuning**

In [15]:
#celda 1
!pip install -q -U transformers peft accelerate evaluate rouge_score datasets torchao

import json
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

MODEL_NAME = "google/flan-t5-small"
PREFIJO_TAREA = "redacta una recomendación de viaje con estos datos: "
MAX_INPUT_LEN = 220  # subido de 64: el input ahora incluye el bloque de hechos (clima, vuelo, lugares)
MAX_TARGET_LEN = 256

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Usando:", device)

Usando: cuda


In [16]:
#celda2
dataset = load_dataset("json", data_files={
    "train": "data/07_train.jsonl",
    "validation": "data/07_val.jsonl",
})
print(dataset)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocesar(ejemplo):
    entrada = PREFIJO_TAREA + ejemplo["input"]
    modelo_in = tokenizer(entrada, max_length=MAX_INPUT_LEN, truncation=True)
    etiquetas = tokenizer(text_target=ejemplo["output"], max_length=MAX_TARGET_LEN, truncation=True)
    modelo_in["labels"] = etiquetas["input_ids"]
    return modelo_in

dataset_tok = dataset.map(preprocesar, remove_columns=["input", "output"])

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output'],
        num_rows: 276
    })
    validation: Dataset({
        features: ['input', 'output'],
        num_rows: 70
    })
})


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

...

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

Map:   0%|          | 0/276 [00:00<?, ? examples/s]

Map:   0%|          | 0/70 [00:00<?, ? examples/s]

In [17]:
#Celda 3
rouge = evaluate.load("rouge")

def cargar_val_crudo(path="data/07_val.jsonl"):
    ejemplos = []
    with open(path, encoding="utf-8") as f:
        for linea in f:
            ejemplos.append(json.loads(linea))
    return ejemplos

def evaluar_modelo(modelo, ejemplos, etiqueta=""):
    modelo.eval()
    predicciones, referencias = [], []
    with torch.no_grad():
        for ej in ejemplos:
            entrada = PREFIJO_TAREA + ej["input"]
            ids = tokenizer(entrada, return_tensors="pt", max_length=MAX_INPUT_LEN, truncation=True).to(device)
            salida = modelo.generate(
                **ids,
                max_length=MAX_TARGET_LEN,
                min_new_tokens=60,        # fuerza a no cortar demasiado corto
                num_beams=4,
                no_repeat_ngram_size=3,   # prohíbe repetir el mismo trigrama (frena los loops)
                repetition_penalty=1.3,   # penaliza reusar tokens ya generados
                early_stopping=True,
            )
            texto = tokenizer.decode(salida[0], skip_special_tokens=True)
            predicciones.append(texto)
            referencias.append(ej["output"])

    resultado = rouge.compute(predictions=predicciones, references=referencias)
    print(f"\\nROUGE ({etiqueta}):")
    for k, v in resultado.items():
        print(f"  {k}: {v:.4f}")
    return resultado, predicciones, referencias

modelo_base = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
val_ejemplos = cargar_val_crudo()

print("=== Evaluando BASELINE (zero-shot, sin fine-tuning) ===")
rouge_baseline, preds_baseline, refs = evaluar_modelo(modelo_base, val_ejemplos, etiqueta="baseline zero-shot")

model.safetensors: reconstructing file:   0%|          |  0.00B /  308MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

=== Evaluando BASELINE (zero-shot, sin fine-tuning) ===
\nROUGE (baseline zero-shot):
  rouge1: 0.3526
  rouge2: 0.1992
  rougeL: 0.2863
  rougeLsum: 0.2863


In [18]:
#Celda 4
# Justificación de hiperparámetros (para el README):
# - r=8: rank bajo, apropiado para un dataset pequeño (78 ejemplos train) —
#   un rank alto arriesga sobreajustar memorizando las plantillas.
# - lora_alpha=16 (2x el rank): heurística estándar de la literatura de LoRA.
# - target_modules=["q","v"]: proyecciones de atención query/value del
#   encoder-decoder T5 — es el punto recomendado por el paper original de
#   LoRA para mantener pocos parámetros entrenables sin perder capacidad
#   de adaptación.
# - learning_rate=3e-4, epochs=8: la primera corrida con 1e-3 y 15 épocas
#   produjo sobreajuste severo (el modelo colapsó a repetir la palabra más
#   frecuente del dataset, ej. "clima, clima, clima..."). Bajar LR y épocas
#   es la respuesta directa a ese diagnóstico, no un valor arbitrario.
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q", "v"],
)

modelo_lora = get_peft_model(AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device), lora_config)
modelo_lora.print_trainable_parameters()

data_collator = DataCollatorForSeq2Seq(tokenizer, model=modelo_lora)

args = Seq2SeqTrainingArguments(
    output_dir="./travelgenie-lora",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-4,           # punto medio: 1e-3 colapsó en loops, 3e-4 subaprendió
    num_train_epochs=12,          # idem, punto medio entre 15 (sobreajuste) y 8 (corto)
    weight_decay=0.01,            # regularización extra
    eval_strategy="epoch",
    save_strategy="no",
    predict_with_generate=True,
    logging_steps=5,
    seed=SEED,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=modelo_lora,
    args=args,
    train_dataset=dataset_tok["train"],
    eval_dataset=dataset_tok["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

trainer.train()

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


trainable params: 344,064 || all params: 77,305,216 || trainable%: 0.4451


Epoch,Training Loss,Validation Loss
1,1.964682,1.581593
2,1.397720,0.988865
3,0.966129,0.574664
4,0.704654,0.370864
5,0.573811,0.276010
6,0.469382,0.196514
7,0.390756,0.144310
8,0.354587,0.112726
9,0.312486,0.094348
10,0.286229,0.079988


TrainOutput(global_step=420, training_loss=0.75036313022886, metrics={'train_runtime': 129.4075, 'train_samples_per_second': 25.594, 'train_steps_per_second': 3.246, 'total_flos': 249000884379648.0, 'train_loss': 0.75036313022886, 'epoch': 12.0})

In [19]:
#Celda 5
print("=== Evaluando MODELO CON FINE-TUNING (LoRA) ===")
rouge_lora, preds_lora, _ = evaluar_modelo(modelo_lora, val_ejemplos, etiqueta="con fine-tuning LoRA")

print("\\n=== TABLA COMPARATIVA (para el README) ===")
print(f"{'Métrica':<12}{'Baseline':<12}{'Con LoRA':<12}{'Delta':<10}")
for k in rouge_baseline:
    base, lora = rouge_baseline[k], rouge_lora[k]
    print(f"{k:<12}{base:<12.4f}{lora:<12.4f}{lora-base:+.4f}")

print("\\n=== 3 EJEMPLOS CUALITATIVOS ===")
for i in range(3):
    print(f"\\n--- Ejemplo {i+1} ---")
    print("INPUT:    ", val_ejemplos[i]["input"])
    print("BASELINE: ", preds_baseline[i])
    print("LORA:     ", preds_lora[i])
    print("REAL:     ", val_ejemplos[i]["output"])

=== Evaluando MODELO CON FINE-TUNING (LoRA) ===
\nROUGE (con fine-tuning LoRA):
  rouge1: 0.6041
  rouge2: 0.4579
  rougeL: 0.5156
  rougeLsum: 0.5153
\n=== TABLA COMPARATIVA (para el README) ===
Métrica     Baseline    Con LoRA    Delta     
rouge1      0.3526      0.6041      +0.2515
rouge2      0.1992      0.4579      +0.2587
rougeL      0.2863      0.5156      +0.2293
rougeLsum   0.2863      0.5153      +0.2290
\n=== 3 EJEMPLOS CUALITATIVOS ===
\n--- Ejemplo 1 ---
INPUT:     Datos: Destino: Riohacha, La Guajira. Mejor mes por clima: diciembre. Vuelo más económico: agosto desde BOG, $223,222 COP. Hoteles: Castillo del Mar, RIOHACHA, casa. Restaurantes: Los Montaditos, Wow Pizza, Asadero Emir. Atractivos: Muelle peatonal, Comunidad La Raya. Pregunta: Quiero ir a Riohacha, ¿cuándo me conviene viajar?
BASELINE:  recomendación de viaje con estas datos: Datos: Destino: Riohacha, La Guajira. Mejor mes por clima: diciembre. Vuelo más económico: agosto desde BOG, $223,222 COP. Hoteles: Cast